# Glow — actnorm, invertible 1x1 convolutions, and affine coupling

> Tutorial pair for [`glow.py`](glow.py).

## 1. Intuition
RealNVP shuffles coordinates between coupling layers with a *fixed* mask. Glow
asks: why fix the permutation? Replace it with a **learned, invertible linear
mixing** of the channels -- an invertible $1\times1$ convolution -- so the model
itself decides how to route information. Glow also swaps batchnorm for
**actnorm**, a per-channel affine layer initialized from the first batch. Each
Glow step is *actnorm -> invertible 1x1 conv -> affine coupling*, and the whole
flow still has an exact, cheap log-likelihood.

## 2. Concept (the slide)
- A Glow step = **actnorm** + **invertible 1x1 conv** + **affine coupling**.
- **ActNorm:** $y=(x-\mu)e^{\log s}$, with $\mu,s$ initialized so the first batch
  is zero-mean/unit-variance (data-dependent init). $\log|\det|=\sum\log s$.
- **Invertible 1x1 conv:** $y=Wx$ with learnable $W$; generalizes a permutation.
  $\log|\det|=\#\text{pixels}\cdot\log|\det W|$.
- **Affine coupling:** same triangular-Jacobian trick as RealNVP.
- Exact likelihood via change of variables; sample by running steps backward.

## 3. Math derivation — the three log-dets

A flow's log-likelihood is, by change of variables,
$$\log p_X(x)=\log\mathcal N\big(f(x);0,I\big)+\sum_{k}\log\Big|\det\frac{\partial f_k}{\partial h_{k-1}}\Big|.$$
Each Glow component contributes one term.

**(a) ActNorm.** Per-channel affine $y=(x-\mu)\odot e^{\log s}$. The Jacobian is
diagonal, so
$$\log|\det J_{\text{actnorm}}|=\sum_c \log s_c \quad(\times\text{ spatial size}).$$
$\mu,\log s$ are *initialized from the first minibatch* so the activations start
normalized -- a normalization that, unlike batchnorm, is exact and batch-size
independent at test time.

**(b) Invertible $1\times1$ convolution.** A $1\times1$ conv with weight matrix
$W\in\mathbb R^{C\times C}$ mixes channels at each of the $H\cdot W$ spatial
locations: $y_{ij}=W x_{ij}$. Its Jacobian is block-diagonal with $H\cdot W$
identical blocks $W$, hence
$$\boxed{\,\log\big|\det J_{1\times1}\big|=H\cdot W\cdot\log|\det W|\,}.$$
Initializing $W$ as a random rotation (orthogonal) makes $\det W=\pm1$
(volume-preserving) and guarantees invertibility; the inverse pass uses $W^{-1}$.
For 2-D data here, $H=W=1$, $C=2$, so this is just a learned $2\times2$ mixing with
$\log|\det W|$.

**(c) Affine coupling.** Split, keep half, scale-and-shift the rest:
$y_b=x_b\odot e^{s(x_a)}+t(x_a)$. As in RealNVP the Jacobian is triangular, so
$$\log|\det J_{\text{coupling}}|=\sum_j s(x_a)_j.$$

**Putting it together.** One Glow step adds the three terms; a stack adds across
steps. The training objective is the negative exact log-likelihood
$$\mathcal L=-\frac1N\sum_i\Big[\log\mathcal N\big(f(x_i);0,I\big)+\textstyle\sum_k \log|\det J_k(x_i)|\Big],$$
and sampling runs every step's inverse: coupling$^{-1}$, then $W^{-1}$, then
actnorm$^{-1}$.

## 4. Model — actnorm, invertible 1x1 conv, coupling, and a Glow step

In [ ]:
# ===== actual implementation from glow.py =====
from __future__ import annotations

import numpy as np

SEED = 0

def _inv1x1_logdet_numpy(W, H=1, Wd=1):
    """Total log|det| of an invertible 1x1 conv over an HxWd feature map."""
    return H * Wd * np.log(abs(np.linalg.det(W)))

import torch

import torch.nn as nn

def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

class ActNorm(nn.Module):
    r"""
    Per-dimension affine transform y = (x - mu) * exp(log_s), initialized from the
    first batch so that y has zero mean and unit variance (data-dependent init).
    log|det| = sum(log_s) per example.
    """

    def __init__(self, dim: int):
        super().__init__()
        self.log_s = nn.Parameter(torch.zeros(dim))
        self.bias = nn.Parameter(torch.zeros(dim))
        self.register_buffer("initialized", torch.tensor(0))

    def _init(self, x: torch.Tensor) -> None:
        with torch.no_grad():
            mu = x.mean(0)
            std = x.std(0) + 1e-6
            self.bias.data.copy_(mu)
            self.log_s.data.copy_(-torch.log(std))    # so output has unit std
            self.initialized.fill_(1)

    def forward(self, x: torch.Tensor):
        if self.initialized.item() == 0 and self.training:
            self._init(x)
        y = (x - self.bias) * torch.exp(self.log_s)
        log_det = self.log_s.sum().expand(len(x))
        return y, log_det

    def inverse(self, y: torch.Tensor):
        return y * torch.exp(-self.log_s) + self.bias

class Inv1x1(nn.Module):
    r"""
    Invertible 1x1 convolution = a learned linear mixing y = W x of the
    coordinates. log|det J| = log|det W| (per example). W is initialized as a
    random rotation (orthogonal) so it starts volume-preserving and invertible.
    """

    def __init__(self, dim: int):
        super().__init__()
        q, _ = torch.linalg.qr(torch.randn(dim, dim))   # random orthogonal init
        self.W = nn.Parameter(q)

    def forward(self, x: torch.Tensor):
        y = x @ self.W.t()
        log_det = torch.slogdet(self.W)[1].expand(len(x))
        return y, log_det

    def inverse(self, y: torch.Tensor):
        return y @ torch.inverse(self.W).t()

class AffineCoupling(nn.Module):
    """Affine coupling on the first / second half of the coordinates."""

    def __init__(self, dim: int, mask: torch.Tensor, hidden: int = 64):
        super().__init__()
        self.register_buffer("mask", mask)
        self.net = nn.Sequential(
            nn.Linear(dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 2 * dim))
        self.scale = nn.Parameter(torch.zeros(dim))

    def _st(self, x_a):
        s, t = self.net(x_a).chunk(2, dim=1)
        return torch.tanh(s) * self.scale, t

    def forward(self, x):
        x_a = x * self.mask
        s, t = self._st(x_a)
        s = s * (1 - self.mask); t = t * (1 - self.mask)
        y = x_a + (1 - self.mask) * (x * torch.exp(s) + t)
        return y, s.sum(1)

    def inverse(self, y):
        y_a = y * self.mask
        s, t = self._st(y_a)
        s = s * (1 - self.mask); t = t * (1 - self.mask)
        return y_a + (1 - self.mask) * ((y - t) * torch.exp(-s))

class GlowStep(nn.Module):
    """One Glow step: actnorm -> invertible 1x1 conv -> affine coupling."""

    def __init__(self, dim: int, mask: torch.Tensor, hidden: int = 64):
        super().__init__()
        self.actnorm = ActNorm(dim)
        self.inv1x1 = Inv1x1(dim)
        self.coupling = AffineCoupling(dim, mask, hidden)

    def forward(self, x):
        ld = torch.zeros(len(x), device=x.device)
        x, d = self.actnorm(x); ld = ld + d
        x, d = self.inv1x1(x); ld = ld + d
        x, d = self.coupling(x); ld = ld + d
        return x, ld

    def inverse(self, y):
        y = self.coupling.inverse(y)
        y = self.inv1x1.inverse(y)
        y = self.actnorm.inverse(y)
        return y

class Glow(nn.Module):
    """A small Glow flow over 2-D data with a standard-normal base."""

    def __init__(self, dim: int = 2, n_steps: int = 6, hidden: int = 64):
        super().__init__()
        steps = []
        for i in range(n_steps):
            m = torch.zeros(dim); m[i % 2::2] = 1.0
            steps.append(GlowStep(dim, m, hidden))
        self.steps = nn.ModuleList(steps)
        self.dim = dim

    def forward(self, x):
        ld = torch.zeros(len(x), device=x.device)
        z = x
        for st in self.steps:
            z, d = st(z); ld = ld + d
        return z, ld

    def inverse(self, z):
        x = z
        for st in reversed(self.steps):
            x = st.inverse(x)
        return x

    def log_prob(self, x):
        z, ld = self(x)
        base = -0.5 * (z ** 2 + np.log(2 * np.pi)).sum(1)
        return base + ld

    def fit(self, X, epochs: int = 400, batch: int = 256, lr: float = 5e-3):
        dev = get_device()
        self.to(dev)
        X = torch.as_tensor(X, dtype=torch.float32, device=dev)
        # one forward pass to trigger actnorm data-dependent init
        self.train()
        with torch.no_grad():
            self(X[:batch])
        opt = torch.optim.Adam(self.parameters(), lr=lr)
        self.history = []
        for _ in range(epochs):
            perm = torch.randperm(len(X), device=dev)
            tot = 0.0
            for s in range(0, len(X), batch):
                nll = -self.log_prob(X[perm[s:s + batch]]).mean()
                opt.zero_grad(); nll.backward(); opt.step()
                tot += nll.item()
            self.history.append(tot / max(1, len(X) // batch))
        return self

    @torch.no_grad()
    def sample(self, n: int):
        dev = next(self.parameters()).device
        z = torch.randn(n, self.dim, device=dev)
        return self.inverse(z).cpu().numpy()

def demo():
    np.random.seed(SEED); torch.manual_seed(SEED)
    torch.set_num_threads(1)  # tiny model: 1 thread avoids CPU thrashing
    from sklearn.datasets import make_moons
    X, _ = make_moons(2000, noise=0.05, random_state=SEED)
    X = ((X - X.mean(0)) / X.std(0)).astype(np.float32)

    m = Glow(dim=2, n_steps=6).fit(X, epochs=400)
    ll = m.log_prob(torch.tensor(X)).mean().item()
    print(f"Glow final NLL = {m.history[-1]:.3f}  (mean log-lik = {ll:.3f})")

    s = m.sample(2000)
    print(f"  data    mean={X.mean(0).round(2)}  std={X.std(0).round(2)}")
    print(f"  samples mean={s.mean(0).round(2)}  std={s.std(0).round(2)}")

## 5. Training / sampling — exact log_prob, NLL training, inverse sampler

In [ ]:
# ===== actual implementation from glow.py =====

## 6. Train & sample on 2-D two-moons

In [ ]:
demo()

## 7. Visualization — learned density and samples

In [ ]:
import matplotlib; matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt, torch
from sklearn.datasets import make_moons
import glow as M

X, _ = make_moons(2000, noise=0.05, random_state=0)
X = ((X - X.mean(0)) / X.std(0)).astype("float32")
m = M.Glow(dim=2, n_steps=6).fit(X, epochs=400)

gx, gy = np.meshgrid(np.linspace(-2.5, 2.5, 120), np.linspace(-2.5, 2.5, 120))
grid = np.c_[gx.ravel(), gy.ravel()].astype("float32")
with torch.no_grad():
    logp = m.log_prob(torch.tensor(grid)).numpy().reshape(gx.shape)
s = m.sample(2000)

fig, ax = plt.subplots(1, 2, figsize=(11, 5))
ax[0].contourf(gx, gy, np.exp(logp), levels=30, cmap="magma")
ax[0].scatter(X[:, 0], X[:, 1], s=3, alpha=.2, color="cyan")
ax[0].set_title("Exact learned density (Glow)"); ax[0].set_aspect("equal")
ax[1].scatter(X[:, 0], X[:, 1], s=4, alpha=.3, label="data")
ax[1].scatter(s[:, 0], s[:, 1], s=4, alpha=.4, color="r", label="Glow samples")
ax[1].legend(); ax[1].set_title("Samples (z~N(0,I) run backward)"); ax[1].set_aspect("equal")
plt.tight_layout(); plt.show()

## 8. Takeaways & pitfalls
- Glow = RealNVP + two upgrades: a **learned** invertible $1\times1$ conv instead
  of a fixed permutation, and **actnorm** instead of batchnorm.
- The $1\times1$ conv's log-det is the clean $H W\log|\det W|$; orthogonal init
  keeps it invertible and stable.
- ActNorm's **data-dependent initialization** must happen once on a real batch
  before training (the code triggers it on the first forward pass).
- Pitfalls: $W$ can drift toward singular (det -> 0); the original paper uses an
  LU-parameterization for cheaper, stable log-dets in high channel counts.
- Like RealNVP, dimensionality is preserved -- flows are exact bijections, the
  price for an exact likelihood.